# Semi-Aves 이미지 데이터 기반 Semi-Supervised Learning 실험

CVPR 2020 FGVC7 워크숍의 Semi-Supervised iNaturalist-Aves 챌린지 데이터(조류 200종)로 방법 6종을 비교한다.
세 논문이 원래 대상으로 한 이미지 데이터이므로 증강은 표준 구성(약한 증강: crop·반전 / 강한 증강: RandAugment·Cutout)을 그대로 쓴다.

- 공식 분할 그대로: labeled 3,959장 · unlabeled(같은 종) 26,640장 · val 2,000장 · test 8,000장
- 방법 6종 × seed 3개 = 18회 학습 (한 번에 하나씩 돈다)
- 로직은 `sslimg/*.py`, 방법별 손실은 시계열 실험과 같은 `sslexp/methods.py`. 설명은 `README.md`
- 끊기면 같은 셀을 다시 실행한다(끝난 실행은 건너뛴다)

In [ ]:
# 0. 준비: Drive 연결, 경로, 코드 불러오기
from google.colab import drive
drive.mount('/content/drive')
import os, sys, importlib, subprocess, time
TIMES = {}      # 단계별 소요 시간(초)

CODE_DIR    = "/content/drive/MyDrive/ssl_seminar/ssl_experiment"       # sslexp, sslimg 폴더가 바로 아래에 있는 위치
RAW         = "/content/semi_aves"                                      # 내려받은 원본(런타임 로컬 디스크, 세션이 끝나면 사라진다)
CACHE       = "/content/semi_aves_cache"                                # 학습에 쓰는 uint8 캐시(로컬 디스크가 빠르다)
CACHE_DRIVE = "/content/drive/MyDrive/ssl_seminar/semi_aves_cache"      # 캐시 보관용(다음 세션에서 내려받기를 건너뛴다)
OUT         = "/content/drive/MyDrive/ssl_seminar/results_img_v1"       # 설정을 바꾸면 새 폴더로
SIZE        = 176                                                       # 캐시 이미지의 한 변(학습 crop은 160)

for name in ("sslexp", "sslimg"):
    if not os.path.isdir(f"{CODE_DIR}/{name}"):
        raise FileNotFoundError(f"{CODE_DIR} 아래에 {name} 폴더가 없습니다. CODE_DIR을 고치거나 폴더를 올리세요.")
sys.path.insert(0, CODE_DIR)
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
import sslexp.methods as M, sslexp.run as SR, sslimg.data as D, sslimg.train as T, sslimg.run as R
M, SR, D, T, R = (importlib.reload(m) for m in (M, SR, D, T, R))
HW = SR.hw_profile()
if not HW["cuda"]: print("GPU가 없습니다. 런타임 유형을 GPU로 바꾸세요.")

In [ ]:
# 1. 데이터 준비. Drive에 캐시가 있으면 그것을 로컬로 복사만 하고, 없으면 Kaggle에서 받아(15.35 GB) 캐시(약 3.8 GB)를 만든다.
#    Kaggle에서 받으려면 먼저 두 가지가 필요하다.
#      (1) https://www.kaggle.com/c/semi-inat-2020 에서 "Join the competition"을 눌러 대회 규칙에 동의
#      (2) Kaggle API 토큰(KGAT_...)을 Colab 왼쪽 열쇠(보안 비밀)에 넣고 "노트북 액세스"를 켠다. 토큰을 이 셀에 직접 적지 않는다.
#    직접 받아 둔 폴더가 있으면 ARCHIVES에 그 경로를 적는다(압축을 풀지 않아도 된다).
TOKEN_SECRET = "KAGGLE_API_TOKEN"     # 보안 비밀에 토큰을 넣을 때 쓴 이름. 다른 이름(예: "SSL")으로 넣었으면 여기를 바꾼다
ARCHIVES = None                       # 예: "/content/drive/MyDrive/ssl_seminar/semi_aves_raw"

_t = time.time()
have_cache = os.path.exists(f"{CACHE_DRIVE}/meta_{SIZE}.json")
if have_cache:
    D.copy_cache(CACHE_DRIVE, CACHE)
elif ARCHIVES is None:
    import kagglehub
    tok = None
    try:
        from google.colab import userdata
        try:
            tok = userdata.get(TOKEN_SECRET)
        except Exception as e:
            print(f"보안 비밀 '{TOKEN_SECRET}'을 읽지 못했습니다({type(e).__name__}). 이름과 '노트북 액세스'를 확인하세요.")
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):      # 예전 방식(kaggle.json)의 값이 들어 있으면 그것도 쓴다
            try:
                if userdata.get(k): os.environ[k] = userdata.get(k)
            except Exception:
                pass
    except ImportError:
        pass
    if tok:
        parts = tok.split()
        if len(parts) > 1: print(f"보안 비밀 값에 줄(또는 공백)이 {len(parts)}개 들어 있어 첫 번째만 씁니다. 값에 토큰이 한 번만 들어가게 고쳐 두세요.")
        tok = parts[0]
        os.environ["KAGGLE_API_TOKEN"] = tok
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f: f.write(tok)
        os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
        print("Kaggle 토큰을 보안 비밀에서 읽었습니다.")
    elif not (os.environ.get("KAGGLE_KEY") or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))):
        kagglehub.login()
data = R.prepare(RAW, CACHE, size=SIZE, archive_dir=ARCHIVES, source="kaggle")
D.copy_cache(CACHE, CACHE_DRIVE)
TIMES["데이터 준비"] = time.time() - _t; print(f"[소요 시간] 데이터 준비 {SR.fmt_time(TIMES['데이터 준비'])}")

In [ ]:
# (선택) 학습을 시작하지 않고 예상 소요 시간만 본다.
cfg = T.ImgCfg(iters=4000)        # ResNet-50(ImageNet 사전학습), labeled 배치 16, unlabeled 배치 112, crop 160
_ = R.estimate_time(data, OUT, cfg=cfg)

In [ ]:
# 2. 주 실험. 처음에는 SEEDS = [0], METHODS = ["supervised", "fixmatch"]로 줄여 한 번 돌려 본다.
SEEDS      = [0, 1, 2]
CONDITIONS = ["in"]                   # in = unlabeled가 같은 200종(공식 설정)
METHODS    = list(M.METHODS)          # supervised, pseudolabel, fixmatch, cgmatch, interlude, scssl

_t = time.time()
R.run_grid(data, OUT, methods=METHODS, conditions=CONDITIONS, seeds=SEEDS, cfg=cfg)
TIMES["학습"] = time.time() - _t; print(f"[소요 시간] 학습(이번 셀 실행분) {SR.fmt_time(TIMES['학습'])}")

In [ ]:
# 3. 결과 표 (OUT/tables 에도 저장된다). 공식 지표는 test top-1 정확도.
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
for name, t in R.tables(OUT).items():
    print(f"\n== {name}"); display(t)

# 소요 시간 요약: 이번 세션의 단계별 시간과, 저장된 실행 기록으로 계산한 방법별 학습 시간
print("\n== 소요 시간")
for k, v in TIMES.items(): print(f"{k}: {SR.fmt_time(v)}")
runs = R.collect(OUT)
if len(runs):
    per = runs.groupby("method").minutes.agg(["mean", "sum", "count"]).reindex([m for m in M.METHODS if m in set(runs.method)])
    for m, r in per.iterrows(): print(f"{SR.LABELS[m]:<16} 1회 평균 {r['mean']:.1f}분 x {int(r['count'])}회 = {r['sum']:.0f}분")
    print(f"학습 시간 합계(저장된 {len(runs)}회): {SR.fmt_time(runs.minutes.sum() * 60)}")

## 선택: open-set 조건

unlabeled에 **다른 종**의 이미지를 섞는다(U_out에서 26,640장). 세 논문이 공통으로 전제한 "unlabeled도 같은 클래스에서 나온다"가 깨지는 조건이다.
Kaggle에서 받았다면 U_out도 이미 받아져 있다(압축 풀기와 캐시 만들기만 추가). 18회 학습이 추가된다.

In [ ]:
# U_OUT_N = 26640
# data = R.prepare(RAW, CACHE, size=SIZE, u_out_n=U_OUT_N, archive_dir=ARCHIVES, source="kaggle"); D.copy_cache(CACHE, CACHE_DRIVE)
# R.run_grid(data, OUT, methods=METHODS, conditions=["open"], seeds=SEEDS, cfg=cfg)
# for name, t in R.tables(OUT).items(): print(f"\n== {name}"); display(t)